## Examen  
### Clustering avec scikit-learn

> Les deux principales méthodes de classification non supervisée, vues au cours de la formation, sont la **classification ascendante hiérarchique (CAH)** et les méthodes de **centres mobiles (K-Means)**. Cependant, elles présentent toutes deux certains avantages et inconvénients :
>
> **L'algorithme de K-Means** est facile à mettre en œuvre et applicable à tout type et toute taille de données. Cependant, le nombre de classes doit être fixé *a priori*, et le résultat final dépend du tirage initial (aléatoire) des centres de classes.
>
> **La classification ascendante hiérarchique (CAH)** a pour principal avantage de permettre la visualisation du regroupement progressif des données ainsi que l’augmentation de la dispersion dans un groupe produite par une agrégation, grâce au **dendrogramme**.  
> Il n’est pas nécessaire de définir le nombre de classes à l’avance, et le dendrogramme permet d’estimer un nombre adéquat de classes.  
> En revanche, la CAH nécessite le calcul des distances entre chaque paire d’individus, ce qui peut devenir très coûteux en temps de calcul lorsque le nombre d’individus est élevé (1000+).
>
> Une solution efficace pour bénéficier des avantages de ces deux méthodes tout en limitant leurs inconvénients respectifs est le recours à la **classification mixte**.
>
> Elle se compose de trois étapes :
>
>> 1. Application de la méthode des **K-Means** pour obtenir rapidement un nombre assez élevé de classes homogènes.  
>>    Une bonne pratique consiste à choisir un nombre de groupes environ 10 fois inférieur au nombre d’observations.
>>
>> 2. Application d’une **classification ascendante hiérarchique (CAH)** sur les **centroïdes** obtenus lors de la première étape, en utilisant un dendrogramme pour choisir le nombre de clusters.
>>
>> 3. Application de l’algorithme **K-Means** sur l’ensemble du jeu de données original, **en utilisant les centroïdes issus de la CAH comme centroïdes initiaux**.
>
> L’objectif du test est de mettre en application un algorithme de **classification mixte** à partir des packages et méthodes étudiées dans la formation.  
> Aucune connaissance extérieure au cours n’est nécessaire pour réaliser cette étape.
>
> Les données utilisées proviennent d’images de divers véhicules et contiennent des caractéristiques propres à leur silhouette.  
> Le but de l’exercice est de réussir à **classer** les véhicules en plusieurs groupes selon ces caractéristiques.

---

## 1. Import et préparation des données

> Dans cette partie, nous allons importer les données et les modules utilisés dans l’exercice.  
> Nous explorerons également les données et effectuerons les opérations de nettoyage nécessaires.

**(a)** Exécuter la commande suivante pour importer les modules nécessaires à cet examen.


In [1]:
import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering, KMeans
from scipy.cluster.hierarchy import dendrogram, linkage

import matplotlib.pyplot as plt
%matplotlib inline 

* (b) En utilisant `pandas`, créer un `DataFrame` appelé **`sv_data`** à partir du fichier **`"vehicles_silhouette.csv"`**. **Déterminer la colonne contenant l'index des véhicules**.

In [ ]:
# Insérez votre code ici
sv_data = pd.read_csv("vehicles_silhouette.csv"
                 ,index_col = 'vehicule_id'
                )





* (c) Afficher les 5 premiéres lignes du jeu de données.

In [ ]:
# Insérez votre code ici

sv_data.head()

#---
# l'index des véhicules est contenu dans la colonne 'vehicule_id'


             Compactness  Circularity  Distance_Cecularity  Radius_Ratio  \
vehicule_id                                                                
0                     95           48                   83           178   
1                     91           41                   84           141   
2                    104           50                  106           209   
3                     93           41                   82           159   
4                     85           44                   70           205   

             PR.Axis_Aspect_Ratio  Max.Length_Aspect_Ratio  Scatter_Ratio  \
vehicule_id                                                                 
0                              72                       10            162   
1                              57                        9            149   
2                              66                       10            207   
3                              63                        9            144   
4    

* (d) Afficher une description des différentes variables en utilisant la méthode `describe`.

In [9]:
# Insérez votre code ici
sv_data.describe()



       Compactness  Circularity  Distance_Cecularity  Radius_Ratio  \
count   846.000000   846.000000           846.000000    846.000000   
mean     93.678487    44.861702            82.088652    168.940898   
std       8.234474     6.169866            15.771533     33.472183   
min      73.000000    33.000000            40.000000    104.000000   
25%      87.000000    40.000000            70.000000    141.000000   
50%      93.000000    44.000000            80.000000    167.000000   
75%     100.000000    49.000000            98.000000    195.000000   
max     119.000000    59.000000           112.000000    333.000000   

       PR.Axis_Aspect_Ratio  Max.Length_Aspect_Ratio  Scatter_Ratio  \
count            846.000000               846.000000     846.000000   
mean              61.693853                 8.567376     168.839243   
std                7.888251                 4.601217      33.244978   
min               47.000000                 2.000000     112.000000   
25%           

* (e) A l'aide d'un `boxplot`, comparer visuellement les distributions des variables numériques.

In [20]:
# Insérez votre code ici
liste = []
for col in sv_data.columns:
#     print(sv_data[col][0])
    liste_cols.append(col)
    liste.append(sv_data[col])
plt.figure()
plt.title('Diagramme en boîte')
plt.boxplot(liste, labels = sv_data.columns)
# plt.boxplot(liste[1:3], labels = sv_data.columns[1:3])
plt.show()



* (f) Effectuer une **normalisation Min-Max** sur les différentes variables de `sv_data`. On pourra utiliser la classe `MinMaxScaler` du sous-module `sklearn.preprocessing`.

In [26]:
# Insérez votre code ici
from sklearn.preprocessing import MinMaxScaler

sv_data_normalized = sv_data.copy()

for col in sv_data.columns:
    scaler = MinMaxScaler()
    sv_data_normalized[col] = scaler.fit_transform(sv_data[[col]])
    
    
sv_data_normalized.head()

Compactness  Circularity  Distance_Cecularity  Radius_Ratio  vehicule_id 0 0.478261 0.576923 0.597222 0.323144 1 0.391304 0.307692 0.611111 0.161572 2 0.673913 0.653846 0.916667 0.458515 3 0.434783 0.307692 0.583333 0.240175 4 0.260870 0.423077 0.416667 0.441048 PR.Axis_Aspect_Ratio  Max.Length_Aspect_Ratio  Scatter_Ratio  vehicule_id 0 0.274725 0.150943 0.326797 1 0.109890 0.132075 0.241830 2 0.208791 0.150943 0.620915 3 0.175824 0.132075 0.209150 4 0.615385 0.943396 0.241830 Eongatedness  PR.Axis_Rectagularity  Max.Length_Rectangularity  vehicule_id 0 0.457143 0.250000 0.585714 1 0.542857 0.166667 0.357143 2 0.171429 0.500000 0.571429 3 0.571429 0.166667 0.357143 4 0.542857 0.166667 0.371429 Scaled_Variance_MajorAxis  Scaled_Variance_MinorAxis  vehicule_id 0 0.242105 0.233813 1 0.210526 0.175060 2 0.489474 0.540767 3 0.157895 0.149880 4 0.584211 0.169065 Scaled_Radius_Gyration  Skewness_MajorAXIS  Skewness_MinorAxis  vehicule_id 0 0.471698 0.144737 0.272727 1 0.308176 0.171053 0.409091 2 0.698113 0.184211 0.636364 3 0.113208 0.052632 0.272727 4 0.496855 0.894737 0.409091 Kurtosis_MinorAxis  Kurtosis_MajorAxis  Hollows_Ratio  vehicule_id 0 0.390244 0.366667 0.533333 1 0.341463 0.433333 0.600000 2 0.219512 0.400000 0.500000 3 0.243902 0.766667 0.866667 4 0.268293 0.133333 0.066667

>2. Classification Mixte - 1er KMeans
>
> Dans cette partie, nous allons appliquer la premiére étape de la classfication mixte: nous allons appliquer un algorithme de `KMeans` sur les observations de base. 

* (a) Appliquer un algorithme de `KMeans` sur `sv_data` en prenant **45 clusters**.

In [ ]:
# Insérez votre code ici 
kmeans = KMeans(n_clusters = 45) 
kmeans.fit(sv_data_normalized)

/kernels/python3/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning warnings.warn(

KMeans(n_clusters=45)

* (b) Extraire les **centroïdes** des clusters produits par ce K-Means et les stocker dans une variable nommée **`centroid_kmeans1`**.

In [29]:
# Insérez votre code ici
centroid_kmeans1 = kmeans.cluster_centers_
print(centroid_kmeans1)




[[0.62228261 0.83774038 0.84895833 0.42671943 0.17445055 0.16273585
  0.67034314 0.14285714 0.59895833 0.76830357 0.50740132 0.59779676
  0.70459906 0.17269737 0.13920455 0.13338415 0.36041667 0.51354167]
 [0.36438923 0.20512821 0.42857143 0.15304637 0.11564626 0.09883199
  0.1459695  0.67619048 0.09126984 0.23333333 0.15764411 0.10203266
  0.17100928 0.14160401 0.0952381  0.16144019 0.45555556 0.43174603]
 [0.46583851 0.24175824 0.62599206 0.34684966 0.16954474 0.11051213
  0.38888889 0.35510204 0.29761905 0.23979592 0.3462406  0.3108085
  0.23674753 0.11842105 0.37012987 0.5087108  0.56666667 0.60714286]
 [0.22201665 0.40180033 0.38356974 0.11818266 0.11854103 0.09995986
  0.25003477 0.54832827 0.16666667 0.38237082 0.21959686 0.17477933
  0.42753914 0.30851064 0.12185687 0.26466009 0.13758865 0.10496454]
 [0.75869565 0.73076923 0.93263889 0.44868996 0.17197802 0.16981132
  0.66111111 0.15714286 0.59583333 0.66428571 0.49526316 0.58219424
  0.61918239 0.12960526 0.68181818 0.64146341

* (c) Pour chaque échantillon de `sv_data`, déterminer le cluster auquel il fait partie d'après le K-Means que vous venez d'entrâiner. Stocker les labels produits dans une variable nommée **`kmeans1_predictions`**.

In [31]:
# Insérez votre code ici
kmeans1_predictions = kmeans.labels_
print(kmeans1_predictions)



[17 10 18 31 24 28 13 39 19 41 19 19 11  7  9  0  6 41  0  0 11  6 31  1
  0 15  6  8 35 23 15 31 19 40 33 13  3 12  0 11 35  6 36 44 22 36  1  3
 17 31  6 10  0 31 28 29 41 36  0 37 20 30 37 11  9 30  3  8 15  9  0 34
 42  6 23 20 44  3 18 16 11 35  1 23 44 28 10  6 26  6  0  2 18 17 11 25
  7 11 40  3 24 13 21 21 14 34 40 44  3  9 15 17 23 30  7  7 35  4 16  9
 30  7 31 21 11  7 11 23 27 35 35 43 40 37 25 24 36 39 30 15 33 16 40  3
 27  8 13 10 32 43 20 14 33 40 39 28 33 37 43 13  6 22  2  1  8  8 33 22
  3 37 18 28 27 25 17 44 42 39 33 16  8  6 31 38 35  2 42 13 25 10 22 10
 13  5  2  3 34  3  3  3 17 33 34 34 29 15 15  3 11  0 33  1 10 25  1 44
 16  0 11 21  0 17  9  3 37 31 18 29  4 16  6  6  3  8 39 36 26  3  8 11
 33 27 16  8 36 37 33 42  8 38 37  0 17 39 36  4 33 27 35  0  6 42 21 19
 22  3 16 27 29  6 16  9 10 31 28 39 11 37  4 26 41 11 16  4 17 13 10 15
 35 30  1 24 44 43 28 26  9 30 42  2 25 13 26 19  3 11 40 18 22 40  9 17
 10  4  3 37  1 40  6  5  8 28 16 18 42  3 22  7 31

* (d) Afficher la répartition des observations par cluster. On pourra utiliser la méthode `value_counts` ou un histogramme. Est-ce que la répartition dans les différents clusters vous semble équilibrée?

In [ ]:
# Insérez votre code ici

plt.figure()
plt.hist(kmeans1_predictions, bins=len(set(kmeans1_predictions)))
plt.title("Répartition des échantillons par cluster")
plt.xlabel("Cluster")
plt.ylabel("Nombre d'échantillons")
plt.show()

counts = pd.Series(kmeans1_predictions).value_counts().sort_index()

# Boxplot
plt.figure()
plt.boxplot(counts)
plt.title("Boxplot du nombre d'échantillons par cluster")
plt.ylabel("Nombre d'échantillons")
plt.show()


#la répartition n'est pas très équilibrée : 50% des clusters contiennent entre 15 et 25 échantillons, ce qui constitue déjà une grande disparité (on passe presque du simple au double d'échantillons contenus dans ce 50% des clusters. Alors les 50% de clusters restants contiennent un nombre encore plus différents d'échantillons).




### 3. Classification Mixte - Classification Hiérarchique Ascendante

Dans cette partie, nous allons appliquer une classification ascendante hiérarchique **sur les centroïdes** issus de l'étape précédente.

1. **Objectif** : Regrouper **les centroïdes** obtenus en clusters, et non les échantillons.
2. **Méthode** : Construire un dendrogramme à partir de **`centroid_kmeans1`** pour déterminer un nombre de clusters optimal (**strictement supérieur à 2**).
3. **Variable** : Attribuer ce nombre de clusters à une variable nommée **`n_clusters_cah`**.

In [ ]:
# Initialisaion de la figrue
plt.figure(figsize=(20, 10))

# Génération de la matrice des liens
Z = linkage(centroid_kmeans1, method = 'ward', metric = 'euclidean')

# Affichage du dendrogramme
plt.title("Dendrogramme CAH")
dendrogram(Z, labels = centroid_kmeans1, leaf_rotation = 45., color_threshold = 0)
plt.show()


#Ja choisis de considérer 3 clusters après la visualisation du dendrogramme. Avoir 2 clusters permettrait de mieux séparer les échantillons mais il faut un nombre de clusters supérieur à 2.
# Et en effet, il est intéressant d'avoir plus de 2 clusters. Si on n'avait que 2 clusters, on retrouverait sûrement dans l'un les gros véhicules carrés type camion et dans l'autre, on y trouverait les autres véhicules (à mon avis). Il serait intéressant de prendre assez de clusters pour pouvoir séparer les véhicules en catégories "camion", "van-traffic", "voitures de sport", "citadines", "4x4"...


NameError: name 'plt' is not defined

In [43]:
# Initialisaion de la figrue
plt.figure(figsize=(20, 10))

# Génération de la matrice des liens
Z = linkage(centroid_kmeans1, method = 'ward', metric = 'euclidean')

# Affichage du dendrogramme
plt.title("Dendrogramme CAH")
dendrogram(Z, labels = centroid_kmeans1, leaf_rotation = 45., color_threshold = 0)
plt.show()


/kernels/python3/lib/python3.9/site-packages/matplotlib/text.py:1223: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if s != self._text:


* (b) Appliquer une **classification ascendante hierarchique** sur **`centroid_kmeans1`** en utilisant le nombre de clusters optimal trouvé précédemment. 


* (c) Pour chaque centroïde de **`centroid_kmeans1`**, déterminer le cluster auquel il fait partie d'après la CAH que vous venez d'entrâiner (attribut `labels_`). On stockera le résultat dans une variable nommée **`cah_clusters`**.

In [45]:
# Insérez votre code ici
# Initialisation du classificateur CAH pour 2 clusters
cluster = AgglomerativeClustering(n_clusters = 2)
#je choisis 2 cluster car le dendrogramme montre que lorsqu'on passe de 2 à 1 cluster, les points à l'intérieur du cluster gagnent le plus de distance les uns des autres (donc ne sont le moins regroupables)

# Apprentissage des données 
cluster.fit(centroid_kmeans1)

# Calcul des labels du data set
cah_clusters = cluster.labels_

print(labels)





[1 0 0 0 1 1 0 0 1 1 0 0 0 0 1 0 0 0 1 0 1 0 1 0 0 1 0 0 1 0 0 0 0 0 1 1 0
 0 0 0 1 1 0 0 0]


> Nous devons à présent associer à chaque observation du jeu de départ un cluster issu de la `CAH`. Pour cela, nous devons:
>
> * Associer chacune des observations de `sv_data` à un cluster du K-Means initial.
>
> * **Faire la correspondance entre les clusters du K-Means avec les clusters de la CAH** pour associer un cluster CAH à chaque observation.

* (d) Grâce à **`kmneans1_predictions`** et **`cah_clusters`**, créer une colonne supplémentaire dans **`sv_data`** nommée **`'cluster_CAH'`**. **Cette colonne contiendra le cluster CAH associé**.

In [ ]:
# Insérez votre code ici
# # print(kmeans1_predictions)
# print(cah_clusters)

new_clusters = pd.Series(kmeans1_predictions).map(lambda x : cah_clusters[x])
# print(new_clusters)

# sv_data['cluster_CAH'] = new_clusters

# sv_data.head()


sv_data_normalized['cluster_CAH'] = new_clusters

sv_data_normalized.head()

Compactness  Circularity  Distance_Cecularity  Radius_Ratio  \\nvehicule_id                                                                \\n0               0.478261     0.576923             0.597222      0.323144   \\n1               0.391304     0.307692             0.611111      0.161572   \\n2               0.673913     0.653846             0.916667      0.458515   \\n3               0.434783     0.307692             0.583333      0.240175   \\n4               0.260870     0.423077             0.416667      0.441048   \\n\\n             PR.Axis_Aspect_Ratio  Max.Length_Aspect_Ratio  Scatter_Ratio  \\nvehicule_id                                                                 \\n0                        0.274725                 0.150943       0.326797   \\n1                        0.109890                 0.132075       0.241830   \\n2                        0.208791                 0.150943       0.620915   \\n3                        0.175824                 0.132075       0.209150   \\n4                        0.615385                 0.943396       0.241830   \\n\\n             Eongatedness  PR.Axis_Rectagularity  Max.Length_Rectangularity  \\nvehicule_id                                                                   \\n0                0.457143               0.250000                   0.585714   \\n1                0.542857               0.166667                   0.357143   \\n2                0.171429               0.500000                   0.571429   \\n3                0.571429               0.166667                   0.357143   \\n4                0.542857               0.166667                   0.371429   \\n\\n             Scaled_Variance_MajorAxis  Scaled_Variance_MinorAxis  \\nvehicule_id                                                         \\n0                             0.242105                   0.233813   \\n1                             0.210526                   0.175060   \\n2                             0.489474                   0.540767   \\n3                             0.157895                   0.149880   \\n4                             0.584211                   0.169065   \\n\\n             Scaled_Radius_Gyration  Skewness_MajorAXIS  Skewness_MinorAxis  \\nvehicule_id                                                                   \\n0                          0.471698            0.144737            0.272727   \\n1                          0.308176            0.171053            0.409091   \\n2                          0.698113            0.184211            0.636364   \\n3                          0.113208            0.052632            0.272727   \\n4                          0.496855            0.894737            0.409091   \\n\\n             Kurtosis_MinorAxis  Kurtosis_MajorAxis  Hollows_Ratio  \\nvehicule_id                                                          \\n0                      0.390244            0.366667       0.533333   \\n1                      0.341463            0.433333       0.600000   \\n2                      0.219512            0.400000       0.500000   \\n3                      0.243902            0.766667       0.866667   \\n4                      0.268293            0.133333       0.066667   \\n\\n             cluster_CAH  \\nvehicule_id               \\n0                      0  \\n1                      0  \\n2                      1  \\n3                      0  \\n4                      0

> Nous allons à présent pouvoir calculer les centroïdes de la classification hierarchique ascendante.

* (e) À l'aide de la méthode `groupby`, **calculer les centroïdes de chaque cluster obtenus par CAH**, c'est-à-dire qu'il faudra calculer la **moyenne de chaque colonne** de `sv_data` en fonction de la valeur de la colonne `"cluster_CAH"`. On stockera le résultat dans une variable nommée **`centroids_cah`**.

In [62]:
# Insérez votre code ici

# centroids_cah = sv_data.groupby('cluster_CAH').mean()
# print(centroids_cah)

centroids_cah = sv_data_normalized.groupby('cluster_CAH').mean()
print(centroids_cah)


             Compactness  Circularity  Distance_Cecularity  Radius_Ratio  \
cluster_CAH                                                                
0               0.347432     0.319908             0.452259      0.209147   
1               0.640236     0.710821             0.831685      0.422619   

             PR.Axis_Aspect_Ratio  Max.Length_Aspect_Ratio  Scatter_Ratio  \
cluster_CAH                                                                 
0                        0.155980                 0.115468       0.231320   
1                        0.171727                 0.139687       0.633322   

             Eongatedness  PR.Axis_Rectagularity  Max.Length_Rectangularity  \
cluster_CAH                                                                   
0                0.564791               0.159256                   0.318667   
1                0.168717               0.558757                   0.633801   

             Scaled_Variance_MajorAxis  Scaled_Variance_MinorAxis  \


* (f) Supprimer la colonne `"cluster_CAH"` de `sv_data`.

In [63]:
# Insérez votre code ici

sv_data_normalized = sv_data_normalized.drop('cluster_CAH', axis = 1)
print(sv_data_normalized.head())


             Compactness  Circularity  Distance_Cecularity  Radius_Ratio  \
vehicule_id                                                                
0               0.478261     0.576923             0.597222      0.323144   
1               0.391304     0.307692             0.611111      0.161572   
2               0.673913     0.653846             0.916667      0.458515   
3               0.434783     0.307692             0.583333      0.240175   
4               0.260870     0.423077             0.416667      0.441048   

             PR.Axis_Aspect_Ratio  Max.Length_Aspect_Ratio  Scatter_Ratio  \
vehicule_id                                                                 
0                        0.274725                 0.150943       0.326797   
1                        0.109890                 0.132075       0.241830   
2                        0.208791                 0.150943       0.620915   
3                        0.175824                 0.132075       0.209150   
4    

>4. Classification Mixte - 2e KMeans
>
> Dans cette partie, nous allons réentrainer un algorithme de `KMeans` en précisant cette fois de prendre les centroïdes calculés à l'étape précédente comme centroïdes initiaux.

* (a) Entraîner un algorithme K-Means sur `sv_data` **en passant `centroids_cah` en argument du paramètre `init` de `KMeans`**. **Utiliser le nombre de clusters optimal** trouvé pendant l'étape de la CAH. 

In [64]:
# Insérez votre code ici

kmeans_final = KMeans(
    n_clusters=2,
    init=centroids_cah
#     ,  
#     n_init=1,
#     random_state=42
)
kmeans_final.fit(sv_data_normalized)

/kernels/python3/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/kernels/python3/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:1362: RuntimeWarning: Explicit initial center position passed: performing only one init in KMeans instead of n_init=10.
  super()._check_params_vs_input(X, default_n_init=10)


KMeans(init=             Compactness  Circularity  Distance_Cecularity  Radius_Ratio  \
cluster_CAH                                                                
0               0.347432     0.319908             0.452259      0.209147   
1               0.640236     0.710821             0.831685      0.422619   

             PR.Axis_Aspect_Ratio  Max.Length_Aspect_Ratio  Scatter_Ratio  \
cluster_CAH                                                                 
0                        0.155980                 0.115468       0.231320   
1                        0.171727                 0.139687       0.633322   

             Eongatedness  PR.Axis_Rectagularity  Max.Length_Rectangularity  \
cluster_CAH                                                                   
0                0.564791               0.159256                   0....
1                0.168717               0.558757                   0.633801   

             Scaled_Variance_MajorAxis  Scaled_Variance_MinorAxis  \
cluster_CAH                                                         
0                             0.208988                   0.169574   
1                             0.494523                   0.563248   

             Scaled_Radius_Gyration  Skewness_MajorAXIS  Skewness_MinorAxis  \
cluster_CAH                                                                   
0                          0.306122            0.184521            0.276275   
1                          0.613282            0.163336            0.315254   

             Kurtosis_MinorAxis  Kurtosis_MajorAxis  Hollows_Ratio  
cluster_CAH                                                         
0                      0.283786            0.420750       0.458016  
1                      0.351220            0.450395       0.543277  ,
       n_clusters=2)

* (c) Pour chaque échantillon de `sv_data`, déterminer le cluster auquel il fait partie d'après le K-Means que vous venez d'entrâiner. Stocker les labels produits dans une variable nommée **`kmeans2_predictions`**.

In [ ]:
# Insérez votre code ici




## 5. Prédictions

> Dans un fichier nommé **`"classes_vehicles.csv"`**, nous avons le type de véhicule pour chacune des observation de `sv_data`: **`car`**, **`bus`** ou **`van`**. Nous allons déterminer si nos clusters sont bien représentatifs de ces étiquettes.

* (a) Lire le fichier **`"classes_vehicles.csv"`** dans un `DataFrame` nommé **`classes`** et en afficher les 5 premiéres lignes.

In [ ]:
# Insérez votre code ici




* (b) En utilisant un tableau croisé, comparer les clusters trouvés grâce à la classification mixte avec les types de véhicules. **Déterminer quel cluster pourrait correspondre à chaque type de véhicule**.

In [ ]:
# Insérez votre code ici


